In [2]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Output
from IPython.display import display
import trino

# ==========================================
# 1. DATABASE CONNECTION & QUERY ENGINE
# ==========================================
class AhaMoveTrinoEngine:
    def __init__(self):
        self.host = os.getenv("TRINO_HOST", "trino-coordinator")
        self.port = int(os.getenv("TRINO_PORT", 8080))
        self.user = os.getenv("TRINO_USER", "admin")
        self.catalog = os.getenv("TRINO_CATALOG", "lakehouse")
        self.schema = os.getenv("TRINO_SCHEMA", "gold")

    def get_connection(self):
        return trino.dbapi.connect(
            host=self.host, port=self.port, user=self.user,
            catalog=self.catalog, schema=self.schema
        )

    def execute_query(self, query: str) -> pd.DataFrame:
        try:
            with self.get_connection() as conn:
                cursor = conn.cursor()
                cursor.execute(query)
                cols = [desc[0] for desc in cursor.description]
                return pd.DataFrame(cursor.fetchall(), columns=cols)
        except Exception as e:
            return pd.DataFrame() # Fallback to Mock Data

db_engine = AhaMoveTrinoEngine()

# ==========================================
# 2. BRAND COLOR & UI SETUP
# ==========================================
COLOR_PRIMARY = "#FF6B00"       # AhaMove Orange
COLOR_SECONDARY = "#0B192C"     # Deep Navy
COLOR_SUCCESS = "#2ECC71"       # Green
COLOR_DANGER = "#E74C3C"        # Red
COLOR_WARNING = "#F1C40F"       # Yellow
COLOR_GRAY = "#7F8C8D"          # Slate Gray

# Global Filters
date_start_widget = widgets.DatePicker(description='Từ ngày:', value=datetime.date(2026, 8, 1), layout=Layout(width='30%'))
date_end_widget = widgets.DatePicker(description='Đến ngày:', value=datetime.date(2026, 8, 19), layout=Layout(width='30%'))
service_widget = widgets.Dropdown(options=['ALL', 'BIKE_INSTANT', 'BIKE_SAMEDAY', 'TRUCK'], value='ALL', description='Dịch vụ:', layout=Layout(width='30%'))
partner_widget = widgets.Dropdown(options=['ALL', 'C2C', 'B2B Hub'], value='ALL', description='Đối tác:', layout=Layout(width='30%'))
btn_refresh = widgets.Button(description='Tải dữ liệu', button_style='warning', icon='refresh', layout=Layout(width='15%', margin='0px 0px 0px 10px'))
dashboard_output = Output()

def build_where(s_date, e_date, srv, ptnr):
    wc = [f"d.full_date BETWEEN DATE '{s_date}' AND DATE '{e_date}'"]
    if srv != 'ALL': wc.append(f"f.service_id = '{srv}'")
    return " AND ".join(wc)

# ==========================================
# 3. SECTION 1: TRUE LIQUIDITY & SYSTEM FRAUD
# ==========================================
def render_section_1(s_date, e_date, srv, ptnr):
    where_sql = build_where(s_date, e_date, srv, ptnr)
    
    q1 = f"""
    SELECT 
        d.full_date,
        COUNT(CASE WHEN f.cancel_time IS NULL OR date_diff('second', f.accept_time, f.cancel_time) >= 180 THEN f.order_id END) as valid_accepts,
        COUNT(CASE WHEN f.is_cancelled_before_pickup = TRUE AND date_diff('second', f.accept_time, f.cancel_time) < 180 THEN f.order_id END) as fake_accepts
    FROM gold.fact_orders f JOIN gold.dim_date d ON f.create_date_sk = d.date_sk
    WHERE f.accept_time IS NOT NULL AND {where_sql} GROUP BY d.full_date ORDER BY d.full_date
    """
    df1 = db_engine.execute_query(q1)
    
    if df1.empty:
        dates = pd.date_range(s_date, e_date)
        valid = np.random.randint(15000, 20000, len(dates))
        # Ép Fake Accept tăng vọt > 5% vào 3 ngày cuối để trigger Alert
        fake_rate = np.concatenate([np.random.uniform(0.01, 0.02, len(dates)-3), np.random.uniform(0.055, 0.08, 3)])
        fake = (valid * fake_rate).astype(int)
        df1 = pd.DataFrame({'full_date': dates, 'valid_accepts': valid, 'fake_accepts': fake})

    df1['total'] = df1['valid_accepts'] + df1['fake_accepts']
    df1['fake_pct'] = df1['fake_accepts'] / df1['total'] * 100

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(x=df1['full_date'], y=df1['valid_accepts']/df1['total']*100, name='Valid Accept', marker_color=COLOR_SUCCESS))
    fig1.add_trace(go.Bar(x=df1['full_date'], y=df1['fake_pct'], name='Fake Accept (< 3 mins)', marker_color=COLOR_DANGER))
    fig1.update_layout(barmode='stack', title="<b>1. The 'Fake Accept' Funnel (Đo lường Thanh khoản Ảo)</b>", template="plotly_white", yaxis_title="% Tỷ trọng")
    fig1.add_shape(type="line", x0=df1['full_date'].min(), x1=df1['full_date'].max(), y0=5, y1=5, line=dict(color="red", width=2, dash="dash"))

    latest_fake_pct = df1['fake_pct'].iloc[-1]
    alert = ""
    if latest_fake_pct > 5:
        alert = f"""<div style='background-color:#FADBD8; padding:10px; border-radius:5px;'>
        <b style='color:#78281F;'>🚨 FRAUD ALERT: Tỷ lệ Fake Accept chạm {latest_fake_pct:.1f}% (> Ngưỡng 5%).</b><br>
        <i>Action:</i> Tự động Suspend 24h tài xế có Tỷ lệ Fake Accept > 15%. Yêu cầu Product team cập nhật Rule: Bắt buộc di chuyển >500m hoặc gọi VoIP >15s mới được phép Hủy do lỗi khách hàng.</div>"""
    
    return HTML(alert), fig1

# ==========================================
# 4. SECTION 2: UNIT ECONOMICS & DEAD-MILEAGE
# ==========================================
def render_section_2(s_date, e_date, srv, ptnr):
    # Chart 2: Price Elasticity
    df2 = pd.DataFrame()
    if df2.empty:
        # Mô phỏng đường cong cung giá: Gãy gập (TTA > 120s) khi giá < 5500 VNĐ/km
        rev_km = np.random.uniform(3000, 12000, 500)
        tta = np.where(rev_km < 5500, 
                       120 + np.random.exponential(40, 500), # Rất chậm
                       15 + np.random.exponential(10, 500))  # Rất nhanh
        df2 = pd.DataFrame({'rev_per_km': rev_km, 'tta_seconds': tta})

    fig2 = px.scatter(df2, x='rev_per_km', y='tta_seconds', trendline="lowess", trendline_color_override="red",
                      title="<b>2. Price Elasticity of Acceptance (Tipping point: 5,500 VNĐ/km)</b>")
    fig2.add_vline(x=5500, line_dash="dash", line_color="orange")
    fig2.update_layout(xaxis_title="Đơn giá thực nhận (VNĐ / km)", yaxis_title="Time to Accept (Giây)", template="plotly_white")

    # Chart 3: Dead-Mileage Heatmap (Mô phỏng Mapbox)
    df3 = pd.DataFrame()
    if df3.empty:
        # HCM Center (Q1, Q3) - Nhanh
        lat_c, lng_c = np.random.normal(10.776, 0.02, 300), np.random.normal(106.700, 0.02, 300)
        tta_c = np.random.uniform(10, 45, 300)
        # Binh Chanh / Nha Be - Lâu (Dead-mileage)
        lat_o, lng_o = np.random.normal(10.680, 0.03, 100), np.random.normal(106.600, 0.03, 100)
        tta_o = np.random.uniform(200, 400, 100)
        
        df3 = pd.DataFrame({'lat': np.concatenate([lat_c, lat_o]), 'lng': np.concatenate([lng_c, lng_o]), 'tta': np.concatenate([tta_c, tta_o])})

    fig3 = px.density_mapbox(df3, lat='lat', lon='lng', z='tta', radius=10,
                             center=dict(lat=10.76, lon=106.66), zoom=10, mapbox_style="carto-positron",
                             color_continuous_scale="Reds", title="<b>3. The 'Dead-Mileage' Heatmap (TTA > 300s tại Vùng Ven)</b>")
    
    alert = """<div style='background-color:#FCF3CF; padding:10px; border-radius:5px; margin-top:10px;'>
        <b style='color:#7D6608;'>⚠️ PRICING ACTION: Phát hiện Lỗ đen Nguồn cung (Dead-Mileage) tại Bình Chánh/Nhà Bè.</b><br>
        <i>Action:</i> Áp dụng Destination-based Surge. Bù đắp trực tiếp +25% total_price cho các điểm đến ngoại thành để bù chi phí chạy rỗng chiều về.</div>"""

    return fig2, fig3, HTML(alert)

# ==========================================
# 5. SECTION 3: OPERATIONAL FRICTION (COD & STOPS)
# ==========================================
def render_section_3(s_date, e_date, srv, ptnr):
    # Mock Matrix Data
    cod_buckets = ['Không COD', '< 500k', '500k - 1M', '> 1M']
    stops = ['1 Điểm', '2 Điểm', '>= 3 Điểm']
    
    # Base TTA is ~20s. Increases heavily with COD and Stops.
    matrix = np.array([
        [20, 35, 75, 120],  # 1 Diem
        [45, 60, 110, 155], # 2 Diem
        [80, 105, 170, 245] # >= 3 Diem (Red zone)
    ])
    
    fig4 = px.imshow(matrix, x=cod_buckets, y=stops, text_auto=True, color_continuous_scale='YlOrRd',
                     labels=dict(x="Mức ứng vốn (COD)", y="Số điểm giao (Stops)", color="Avg TTA (s)"),
                     title="<b>4. The COD & Stops Friction Matrix (Điểm nghẽn Vốn & Rủi ro boom hàng)</b>")
    
    alert = """<div style='background-color:#FADBD8; padding:10px; border-radius:5px;'>
        <b style='color:#78281F;'>🚨 ROUTING & FINTECH ACTION: Rào cản Vận hành quá lớn tại ô [COD > 1M & >= 3 Điểm] (TTA > 180s).</b><br>
        <i>DS Rule:</i> Thuật toán Batching cấm ghép > 2 điểm nếu tổng COD > 1,000,000đ.<br>
        <i>Product Ops:</i> Đẩy mạnh Push Notif chiến dịch "Ví trả sau" cho tài xế Part-time.</div>"""
    
    return HTML(alert), fig4

# ==========================================
# 6. SECTION 4: BATCHING ALGORITHM QUALITY
# ==========================================
def render_section_4(s_date, e_date, srv, ptnr):
    dev_buckets = ['0-10%', '10-20%', '20-30%', '30-40%', '> 40%']
    reject_rate = [2, 8, 15, 42, 68] # Vọt lên >40% khi lệch lộ trình >30%
    avg_tta = [15, 25, 40, 120, 200]

    fig5 = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig5.add_trace(go.Bar(x=dev_buckets, y=reject_rate, name="Tỷ lệ Reject/Ignore (%)", marker_color=COLOR_SECONDARY, opacity=0.7), secondary_y=False)
    fig5.add_trace(go.Scatter(x=dev_buckets, y=avg_tta, name="Avg Time to Accept (s)", line=dict(color=COLOR_DANGER, width=3)), secondary_y=True)

    fig5.update_layout(title="<b>5. Deviation Angle & Batching Quality (Giới hạn chịu đựng Quãng đường vòng)</b>", template="plotly_white")
    fig5.update_yaxes(title_text="Tỷ lệ Từ chối (%)", secondary_y=False)
    fig5.update_yaxes(title_text="Time to Accept (Giây)", secondary_y=True)
    
    alert = """<div style='background-color:#EBF5FB; padding:10px; border-radius:5px; margin-top:10px;'>
        <b style='color:#154360;'>⚙️ ALGORITHM TUNING: Tỷ lệ tẩy chay đơn ghép vượt 40% khi quãng đường vòng > 30%.</b><br>
        <i>Action:</i> Áp dụng Hard Penalty vào Module Routing. Tự động Un-batch thành 2 đơn Single nếu distance_to_target_meters > 30% lộ trình chim bay.</div>"""

    return fig5, HTML(alert)

# ==========================================
# 7. DASHBOARD CONTROLLER & RENDER PIPELINE
# ==========================================
def update_dashboard(b=None):
    dashboard_output.clear_output(wait=True)
    with dashboard_output:
        s_date, e_date, srv, ptnr = date_start_widget.value, date_end_widget.value, service_widget.value, partner_widget.value
        
        # Section 1
        al1, fig1 = render_section_1(s_date, e_date, srv, ptnr)
        display(al1)
        display(fig1)
        
        # Section 2 (Đã fix lỗi FigureWidget bằng Output widgets)
        fig2, fig3, al2 = render_section_2(s_date, e_date, srv, ptnr)
        
        out_fig2 = Output(layout=Layout(width='50%'))
        out_fig3 = Output(layout=Layout(width='50%'))
        
        with out_fig2:
            display(fig2)
        with out_fig3:
            display(fig3)
            
        display(HBox([out_fig2, out_fig3]))
        display(al2)
        
        # Section 3
        al3, fig4 = render_section_3(s_date, e_date, srv, ptnr)
        display(al3)
        display(fig4)
        
        # Section 4
        fig5, al4 = render_section_4(s_date, e_date, srv, ptnr)
        display(al4)
        display(fig5)

btn_refresh.on_click(update_dashboard)

# Layout Assembly
filter_bar = HBox([date_start_widget, date_end_widget, service_widget, partner_widget, btn_refresh], 
                  layout=Layout(background_color='#F4F6F7', padding='10px', border_radius='5px', margin='0px 0px 15px 0px'))

header = HTML("<h1 style='color:#FF6B00; text-align:center;'>AHAMOVE SLA PHASE 2: ORDER ➔ ACCEPT MATRIX</h1>")

dashboard_app = VBox([header, filter_bar, dashboard_output])
display(dashboard_app)

update_dashboard()

In [3]:
import os
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Layout, VBox, HBox, HTML, Output
from IPython.display import display
import trino

# ==========================================
# 1. DATABASE CONNECTION & QUERY ENGINE
# ==========================================
class AhaMoveTrinoEngine:
    def __init__(self):
        self.host = os.getenv("TRINO_HOST", "trino-coordinator")
        self.port = int(os.getenv("TRINO_PORT", 8080))
        self.user = os.getenv("TRINO_USER", "admin")
        self.catalog = os.getenv("TRINO_CATALOG", "lakehouse")
        self.schema = os.getenv("TRINO_SCHEMA", "gold")

    def get_connection(self):
        return trino.dbapi.connect(
            host=self.host, port=self.port, user=self.user,
            catalog=self.catalog, schema=self.schema
        )

    def execute_query(self, query: str) -> pd.DataFrame:
        try:
            with self.get_connection() as conn:
                cursor = conn.cursor()
                cursor.execute(query)
                cols = [desc[0] for desc in cursor.description]
                return pd.DataFrame(cursor.fetchall(), columns=cols)
        except Exception as e:
            return pd.DataFrame() # Fallback to Mock Data nếu không có DB

db_engine = AhaMoveTrinoEngine()

# ==========================================
# 2. BRAND COLOR & UI SETUP
# ==========================================
COLOR_PRIMARY = "#FF6B00"       # AhaMove Orange
COLOR_SECONDARY = "#0B192C"     # Deep Navy
COLOR_SUCCESS = "#2ECC71"       # Green
COLOR_DANGER = "#E74C3C"        # Red
COLOR_WARNING = "#F1C40F"       # Yellow
COLOR_GRAY = "#7F8C8D"          # Slate Gray

# Global Filters
date_start_widget = widgets.DatePicker(description='Từ ngày:', value=datetime.date(2026, 8, 1), layout=Layout(width='30%'))
date_end_widget = widgets.DatePicker(description='Đến ngày:', value=datetime.date(2026, 8, 19), layout=Layout(width='30%'))
service_widget = widgets.Dropdown(options=['ALL', 'BIKE_INSTANT', 'BIKE_SAMEDAY', 'TRUCK'], value='ALL', description='Dịch vụ:', layout=Layout(width='30%'))
partner_widget = widgets.Dropdown(options=['ALL', 'C2C', 'B2B Hub'], value='ALL', description='Đối tác:', layout=Layout(width='30%'))
btn_refresh = widgets.Button(description='Tải dữ liệu', button_style='warning', icon='refresh', layout=Layout(width='15%', margin='0px 0px 0px 10px'))
dashboard_output = Output()

def build_where(s_date, e_date, srv, ptnr):
    wc = [f"d.full_date BETWEEN DATE '{s_date}' AND DATE '{e_date}'"]
    if srv != 'ALL': wc.append(f"f.service_id = '{srv}'")
    return " AND ".join(wc)

# ==========================================
# 3. SECTION 1: TRUE LIQUIDITY & SYSTEM FRAUD
# ==========================================
def render_section_1(s_date, e_date, srv, ptnr):
    where_sql = build_where(s_date, e_date, srv, ptnr)
    
    q1 = f"""
    SELECT 
        d.full_date,
        COUNT(CASE WHEN f.cancel_time IS NULL OR date_diff('second', f.accept_time, f.cancel_time) >= 180 THEN f.order_id END) as valid_accepts,
        COUNT(CASE WHEN f.is_cancelled_before_pickup = TRUE AND date_diff('second', f.accept_time, f.cancel_time) < 180 THEN f.order_id END) as fake_accepts
    FROM gold.fact_orders f JOIN gold.dim_date d ON f.create_date_sk = d.date_sk
    WHERE f.accept_time IS NOT NULL AND {where_sql} GROUP BY d.full_date ORDER BY d.full_date
    """
    df1 = db_engine.execute_query(q1)
    
    if df1.empty:
        dates = pd.date_range(s_date, e_date)
        valid = np.random.randint(15000, 20000, len(dates))
        # Ép Fake Accept tăng vọt > 5% vào 3 ngày cuối để trigger Alert
        fake_rate = np.concatenate([np.random.uniform(0.01, 0.02, len(dates)-3), np.random.uniform(0.055, 0.08, 3)])
        fake = (valid * fake_rate).astype(int)
        df1 = pd.DataFrame({'full_date': dates, 'valid_accepts': valid, 'fake_accepts': fake})

    df1['total'] = df1['valid_accepts'] + df1['fake_accepts']
    df1['fake_pct'] = df1['fake_accepts'] / df1['total'] * 100

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(x=df1['full_date'], y=df1['valid_accepts']/df1['total']*100, name='Valid Accept', marker_color=COLOR_SUCCESS))
    fig1.add_trace(go.Bar(x=df1['full_date'], y=df1['fake_pct'], name='Fake Accept (< 3 mins)', marker_color=COLOR_DANGER))
    fig1.update_layout(barmode='stack', title="<b>1. The 'Fake Accept' Funnel (Đo lường Thanh khoản Ảo)</b>", template="plotly_white", yaxis_title="% Tỷ trọng")
    fig1.add_shape(type="line", x0=df1['full_date'].min(), x1=df1['full_date'].max(), y0=5, y1=5, line=dict(color="red", width=2, dash="dash"))

    latest_fake_pct = df1['fake_pct'].iloc[-1]
    alert = ""
    if latest_fake_pct > 5:
        alert = f"""<div style='background-color:#FADBD8; padding:10px; border-radius:5px;'>
        <b style='color:#78281F;'>🚨 FRAUD ALERT: Tỷ lệ Fake Accept chạm {latest_fake_pct:.1f}% (> Ngưỡng 5%).</b><br>
        <i>Action:</i> Tự động Suspend 24h tài xế có Tỷ lệ Fake Accept > 15%. Yêu cầu Product team cập nhật Rule: Bắt buộc di chuyển >500m hoặc gọi VoIP >15s mới được phép Hủy do lỗi khách hàng.</div>"""
    
    return HTML(alert), fig1

# ==========================================
# 4. SECTION 2: UNIT ECONOMICS & DEAD-MILEAGE
# ==========================================
def render_section_2(s_date, e_date, srv, ptnr):
    # Chart 2: Price Elasticity
    df2 = pd.DataFrame()
    if df2.empty:
        # Mô phỏng đường cong cung giá: Gãy gập (TTA > 120s) khi giá < 5500 VNĐ/km
        rev_km = np.random.uniform(3000, 12000, 500)
        tta = np.where(rev_km < 5500, 
                       120 + np.random.exponential(40, 500), # Rất chậm
                       15 + np.random.exponential(10, 500))  # Rất nhanh
        df2 = pd.DataFrame({'rev_per_km': rev_km, 'tta_seconds': tta})

    fig2 = px.scatter(df2, x='rev_per_km', y='tta_seconds', trendline="lowess", trendline_color_override="red",
                      title="<b>2. Price Elasticity of Acceptance</b><br>(Tipping point: 5,500 VNĐ/km)")
    fig2.add_vline(x=5500, line_dash="dash", line_color="orange")
    fig2.update_layout(xaxis_title="Đơn giá thực nhận (VNĐ / km)", yaxis_title="Time to Accept (Giây)", template="plotly_white", margin=dict(t=50))

    # Chart 3: Dead-Mileage Heatmap (Mô phỏng Mapbox)
    df3 = pd.DataFrame()
    if df3.empty:
        # HCM Center (Q1, Q3) - Nhanh
        lat_c, lng_c = np.random.normal(10.776, 0.02, 300), np.random.normal(106.700, 0.02, 300)
        tta_c = np.random.uniform(10, 45, 300)
        # Binh Chanh / Nha Be - Lâu (Dead-mileage) -> cluster màu đỏ
        lat_o, lng_o = np.random.normal(10.680, 0.03, 100), np.random.normal(106.600, 0.03, 100)
        tta_o = np.random.uniform(200, 400, 100)
        
        df3 = pd.DataFrame({'lat': np.concatenate([lat_c, lat_o]), 'lng': np.concatenate([lng_c, lng_o]), 'tta': np.concatenate([tta_c, tta_o])})

    fig3 = px.density_mapbox(df3, lat='lat', lon='lng', z='tta', radius=12,
                             center=dict(lat=10.76, lon=106.66), zoom=10, mapbox_style="carto-positron",
                             color_continuous_scale="Reds", title="<b>3. Dead-Mileage Heatmap</b><br>(TTA > 300s tại Vùng Ven)")
    fig3.update_layout(margin=dict(t=50))
    
    alert = """<div style='background-color:#FCF3CF; padding:10px; border-radius:5px; margin-top:10px;'>
        <b style='color:#7D6608;'>⚠️ PRICING ACTION: Phát hiện Lỗ đen Nguồn cung (Dead-Mileage) tại Bình Chánh/Nhà Bè.</b><br>
        <i>Action:</i> Áp dụng Destination-based Surge. Bù đắp trực tiếp +25% total_price cho các điểm đến ngoại thành để bù chi phí chạy rỗng chiều về.</div>"""

    return fig2, fig3, HTML(alert)

# ==========================================
# 5. SECTION 3: OPERATIONAL FRICTION (COD & STOPS)
# ==========================================
def render_section_3(s_date, e_date, srv, ptnr):
    # Mock Matrix Data - Mô phỏng logic khi COD và số điểm dừng tăng cao
    cod_buckets = ['Không COD', '< 500k', '500k - 1M', '> 1M']
    stops = ['1 Điểm', '2 Điểm', '>= 3 Điểm']
    
    # Base TTA ~20s. Nhảy vọt lên >180s ở vùng đỏ.
    matrix = np.array([
        [20, 35, 75, 120],  # 1 Điểm
        [45, 60, 110, 155], # 2 Điểm
        [80, 105, 170, 245] # >= 3 Điểm (Red zone)
    ])
    
    fig4 = px.imshow(matrix, x=cod_buckets, y=stops, text_auto=True, color_continuous_scale='YlOrRd',
                     labels=dict(x="Mức ứng vốn (COD)", y="Số điểm giao (Stops)", color="Avg TTA (s)"),
                     title="<b>4. The COD & Stops Friction Matrix (Điểm nghẽn Vốn & Rủi ro boom hàng)</b>")
    
    alert = """<div style='background-color:#FADBD8; padding:10px; border-radius:5px;'>
        <b style='color:#78281F;'>🚨 ROUTING & FINTECH ACTION: Rào cản Vận hành quá lớn tại ô [COD > 1M & >= 3 Điểm] (TTA > 180s).</b><br>
        <i>DS Rule:</i> Thuật toán Batching cấm ghép > 2 điểm nếu tổng COD > 1,000,000đ.<br>
        <i>Product Ops:</i> Đẩy mạnh Push Notif chiến dịch "Ví trả sau" (hợp tác Ngân hàng) cho tập tài xế rớt đơn do thiếu vốn.</div>"""
    
    return HTML(alert), fig4

# ==========================================
# 6. SECTION 4: BATCHING ALGORITHM QUALITY
# ==========================================
def render_section_4(s_date, e_date, srv, ptnr):
    dev_buckets = ['0-10%', '10-20%', '20-30%', '30-40%', '> 40%']
    reject_rate = [2, 8, 15, 42, 68] # Tỷ lệ Reject vọt lên >40% khi lệch >30% lộ trình gốc
    avg_tta = [15, 25, 40, 120, 200]

    fig5 = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig5.add_trace(go.Bar(x=dev_buckets, y=reject_rate, name="Tỷ lệ Reject/Ignore (%)", marker_color=COLOR_SECONDARY, opacity=0.7), secondary_y=False)
    fig5.add_trace(go.Scatter(x=dev_buckets, y=avg_tta, name="Avg Time to Accept (s)", line=dict(color=COLOR_DANGER, width=3)), secondary_y=True)

    fig5.update_layout(title="<b>5. Deviation Angle & Batching Quality (Giới hạn chịu đựng Quãng đường vòng)</b>", template="plotly_white")
    fig5.update_yaxes(title_text="Tỷ lệ Từ chối (%)", secondary_y=False)
    fig5.update_yaxes(title_text="Time to Accept (Giây)", secondary_y=True)
    
    alert = """<div style='background-color:#EBF5FB; padding:10px; border-radius:5px; margin-top:10px;'>
        <b style='color:#154360;'>⚙️ ALGORITHM TUNING: Tỷ lệ tẩy chay đơn ghép vượt 40% khi quãng đường vòng > 30%.</b><br>
        <i>Action:</i> Áp dụng Hard Penalty vào Module Routing. Tự động Un-batch thành 2 đơn Single nếu distance_to_target_meters vượt quá 30% lộ trình chim bay.</div>"""

    return fig5, HTML(alert)

# ==========================================
# 7. DASHBOARD CONTROLLER & RENDER PIPELINE
# ==========================================
def update_dashboard(b=None):
    dashboard_output.clear_output(wait=True)
    with dashboard_output:
        s_date, e_date, srv, ptnr = date_start_widget.value, date_end_widget.value, service_widget.value, partner_widget.value
        
        # Section 1
        al1, fig1 = render_section_1(s_date, e_date, srv, ptnr)
        display(al1)
        display(fig1)
        
        # Section 2 (Fix lỗi bằng widgets.Output)
        fig2, fig3, al2 = render_section_2(s_date, e_date, srv, ptnr)
        
        out_fig2 = Output(layout=Layout(width='50%'))
        out_fig3 = Output(layout=Layout(width='50%'))
        
        with out_fig2:
            display(fig2)
        with out_fig3:
            display(fig3)
            
        display(HBox([out_fig2, out_fig3]))
        display(al2)
        
        # Section 3
        al3, fig4 = render_section_3(s_date, e_date, srv, ptnr)
        display(al3)
        display(fig4)
        
        # Section 4
        fig5, al4 = render_section_4(s_date, e_date, srv, ptnr)
        display(al4)
        display(fig5)

btn_refresh.on_click(update_dashboard)

# Layout Assembly
filter_bar = HBox([date_start_widget, date_end_widget, service_widget, partner_widget, btn_refresh], 
                  layout=Layout(background_color='#F4F6F7', padding='10px', border_radius='5px', margin='0px 0px 15px 0px'))

header = HTML("<h1 style='color:#FF6B00; text-align:center;'>AHAMOVE SLA PHASE 2: ORDER ➔ ACCEPT MATRIX</h1>")

dashboard_app = VBox([header, filter_bar, dashboard_output])
display(dashboard_app)

update_dashboard()